In [1]:
import pandas as pd
import numpy as np

from sklearn.experimental import enable_iterative_imputer #A
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor

In [2]:
df_matches = pd.read_csv('../../data/preprocessed/preprocessed_1.csv')
df_matches.sort_values(by=["season", "stage", "date"], inplace=True)
season_labels, _ = pd.factorize(df_matches['result_match'])
feature_cols = [col for col in df_matches.columns if col not in
                ["result_match", "stage", "season" ,"date", "home_team", "away_team"]]

df = df_matches[["match_api_id", "result_match", "stage", "season", "date", "home_team", "away_team"]]

df_matches = df_matches[feature_cols]
df_matches

,match_api_id,points_home,points_away,home_team_last_goal,home_team_last_shoton,home_team_last_possession,away_team_last_goal,away_team_last_shoton,away_team_last_possession,avg_team_strength_home,...,rolling_stability_shoton_away,shot_efficiency_ratio_home,shot_efficiency_ratio_away,rolling_avg_goals_diff,rolling_stability_goal_diff,rolling_avg_goals_conversion_rate_diff,rolling_stability_goals_conversion_rate_diff,rolling_avg_shoton_diff,rolling_stability_shoton_diff,points_diff
0,489063,3,4,1.00,12.0,34.0,1.000000,1.0,55.0,78.318182,...,NaN,NaN,NaN,0.000000,NaN,-0.916667,NaN,11.000000,NaN,-1
1,489068,3,3,1.00,7.0,47.0,3.000000,1.0,47.0,63.924242,...,NaN,NaN,NaN,-2.000000,NaN,-2.857143,NaN,6.000000,NaN,0
2,489069,3,0,3.00,5.0,53.0,1.000000,2.0,66.0,71.318182,...,NaN,NaN,NaN,2.000000,NaN,0.100000,NaN,3.000000,NaN,3
3,489070,4,0,1.00,5.0,47.0,1.000000,7.0,52.0,63.318182,...,NaN,NaN,NaN,0.000000,NaN,0.057143,NaN,-2.000000,NaN,4
4,489066,3,6,2.00,5.0,48.0,2.000000,11.0,46.0,77.030303,...,NaN,NaN,NaN,0.000000,NaN,0.218182,NaN,-6.000000,NaN,-3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2970,1987599,44,34,1.00,4.0,44.0,4.000000,6.0,69.0,72.327273,...,2.501802,0.221394,0.413129,-1.877456,-0.375831,0.071750,0.604273,-1.750375,0.469080,10
2971,1987598,49,80,2.00,5.0,53.0,3.000000,4.0,67.0,65.445455,...,2.081047,0.563487,0.584040,0.098747,-0.282355,-0.014761,-0.194222,-0.402411,-0.425180,-31
2972,1987597,68,17,2.00,7.0,31.0,1.666667,1.0,65.0,70.627273,...,3.830061,0.228152,0.117402,0.296724,0.415464,-0.535126,-0.536706,3.822830,-0.038204,51
2973,1987601,34,70,1.75,6.0,50.0,1.000000,6.0,47.0,79.690909,...,2.072934,0.249170,0.530648,-0.123565,-0.424164,0.329249,0.205247,-2.467804,0.639423,-36


In [3]:
(
    df_matches.isna()
    .sum(axis=0)
)

match_api_id                                     0
points_home                                      0
points_away                                      0
home_team_last_goal                              0
home_team_last_shoton                            0
home_team_last_possession                        0
away_team_last_goal                              0
away_team_last_shoton                            0
away_team_last_possession                        0
avg_team_strength_home                           0
avg_team_strength_away                           0
strength_difference                              0
avg_team_aggression_home                         0
avg_team_aggression_away                         0
aggression_difference                            0
avg_team_acceleration_home                       0
avg_team_acceleration_away                       0
acceleration_difference                          0
goal_conversion_rate_home                        0
goal_conversion_rate_away      

In [9]:
df_matches[['rolling_stability_shoton_diff','rolling_stability_goals_conversion_rate_diff','rolling_stability_goal_diff','rolling_stability_goal_home','shot_efficiency_ratio_home','rolling_stability_goals_conversion_rate_home','rolling_stability_shoton_home','rolling_stability_goal_away','rolling_stability_goals_conversion_rate_away','rolling_stability_shoton_away','shot_efficiency_ratio_away']].describe().round(2)


,rolling_stability_shoton_diff,rolling_stability_goals_conversion_rate_diff,rolling_stability_goal_diff,rolling_stability_goal_home,shot_efficiency_ratio_home,rolling_stability_goals_conversion_rate_home,rolling_stability_shoton_home,rolling_stability_goal_away,rolling_stability_goals_conversion_rate_away,rolling_stability_shoton_away,shot_efficiency_ratio_away
count,2948.00,2948.00,2948.00,2955.00,2955.00,2955.00,2955.00,2961.00,2961.00,2961.00,2961.00
mean,0.00,-0.00,-0.02,0.78,72.10,0.33,2.84,0.80,0.33,2.84,30.82
std,1.57,0.36,0.59,0.41,2908.25,0.24,1.11,0.42,0.26,1.10,1659.85
min,-6.96,-3.43,-3.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,-0.93,-0.16,-0.35,0.50,0.18,0.17,2.10,0.52,0.17,2.08,0.18
50%,0.00,-0.00,-0.01,0.70,0.27,0.25,2.65,0.71,0.26,2.65,0.27
75%,0.92,0.16,0.32,0.97,0.39,0.40,3.37,0.99,0.39,3.39,0.40
max,6.60,2.80,2.85,3.39,141421.36,3.08,9.16,3.49,4.01,10.51,90321.06


In [4]:
simple_imputer = SimpleImputer(fill_value=-1)
Xm_si = simple_imputer.fit_transform(df_matches)

rf = RandomForestRegressor(random_state=0, n_jobs=-1)
multivariate_imputer = IterativeImputer(estimator=rf, max_iter=10, tol=0.01)
Xm_ii = multivariate_imputer.fit_transform(df_matches)

imputed_si_df = pd.DataFrame(Xm_si, columns=df_matches.columns)
imputed_ii_df = pd.DataFrame(Xm_ii, columns=df_matches.columns)

In [5]:
(imputed_si_df.isna().sum(axis=0))

match_api_id                                    0
points_home                                     0
points_away                                     0
home_team_last_goal                             0
home_team_last_shoton                           0
home_team_last_possession                       0
away_team_last_goal                             0
away_team_last_shoton                           0
away_team_last_possession                       0
avg_team_strength_home                          0
avg_team_strength_away                          0
strength_difference                             0
avg_team_aggression_home                        0
avg_team_aggression_away                        0
aggression_difference                           0
avg_team_acceleration_home                      0
avg_team_acceleration_away                      0
acceleration_difference                         0
goal_conversion_rate_home                       0
goal_conversion_rate_away                       0


In [6]:
(imputed_ii_df.isna().sum(axis=0))

match_api_id                                    0
points_home                                     0
points_away                                     0
home_team_last_goal                             0
home_team_last_shoton                           0
home_team_last_possession                       0
away_team_last_goal                             0
away_team_last_shoton                           0
away_team_last_possession                       0
avg_team_strength_home                          0
avg_team_strength_away                          0
strength_difference                             0
avg_team_aggression_home                        0
avg_team_aggression_away                        0
aggression_difference                           0
avg_team_acceleration_home                      0
avg_team_acceleration_away                      0
acceleration_difference                         0
goal_conversion_rate_home                       0
goal_conversion_rate_away                       0


In [7]:
imputed_ii_df=pd.merge(df, imputed_ii_df, on="match_api_id")

In [10]:
imputed_ii_df.head(20).to_csv('../../data/preprocessed/imputed_data.csv', index=False)